In [ ]:
!pip install -q -U bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.1/59.1 MB 18.1 MB/s eta 0:00:00


In [ ]:

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM,BitsAndBytesConfig
from peft import PeftModel

# Your fine-tuned adapter on Hugging Face
adapter_model_id = "prizmaweb/bodmas-math-2025-12-18_12.10.42-lite"

# IMPORTANT: use the same base model you fine-tuned against
BASE_MODEL = "meta-llama/Llama-3.2-3B"

# Define quantization config
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True, # or load_in_8bit=True
    bnb_4bit_use_double_quant=True, # optional, improves efficiency
    bnb_4bit_quant_type="nf4", # recommended quantization type
    bnb_4bit_compute_dtype=torch.float16 # compute dtype
    )



# Load tokenizer from the base model
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)

# Load base model in 4-bit (QLoRA style)
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    device_map="auto",
    quantization_config=bnb_config
)

# Attach your LoRA adapter
model = PeftModel.from_pretrained(base_model, adapter_model_id)


tokenizer_config.json:   0%|          | 0.00/50.5k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/301 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/844 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/20.9k [00:00<?, ?B/s]

...

model-00002-of-00002.safetensors:   0%|          | 0.00/1.46G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/185 [00:00<?, ?B/s]

adapter_config.json:   0%|          | 0.00/1.00k [00:00<?, ?B/s]

adapter_model.safetensors:   0%|          | 0.00/36.7M [00:00<?, ?B/s]

In [ ]:
# Example inference
# 1. Define the sentence you want to extract entities from
sentence = "Solve using BODMAS: (18 - 2) / (4 * 2)"

# 2. Tokenize and run inference
inputs = tokenizer(sentence, return_tensors="pt").to("cuda")

outputs = model.generate(**inputs, max_new_tokens=100)

# 3. Decode the output back into text
print(tokenizer.decode(outputs[0], skip_special_tokens=True))


Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


Solve using BODMAS: (18 - 2) / (4 * 2)Step 1: Brackets → (18 - 2) = 16, (4 * 2) = 8
Step 2: Division → 16 / 8 = 2
Final Answer: 2


In [ ]:
# Example inference
# 1. Define the sentence you want to extract entities from
sentence = "Solve using BODMAS: (12 * 5)/3 + 4"

# 2. Tokenize and run inference
inputs = tokenizer(sentence, return_tensors="pt").to("cuda")

outputs = model.generate(**inputs, max_new_tokens=100)

# 3. Decode the output back into text
print(tokenizer.decode(outputs[0], skip_special_tokens=True))

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


Solve using BODMAS: (12 * 5)/3 + 4Step 1: Brackets → (12 * 5) = 60
Step 2: Division → 60/3 = 20
Step 3: Addition → 20 + 4 = 24
Final Answer: 24


In [ ]:
# Example inference
# 1. Define the sentence you want to extract entities from
sentence = "Solve using BODMAS: 8 + 3 * (6 - 2)/4"

# 2. Tokenize and run inference
inputs = tokenizer(sentence, return_tensors="pt").to("cuda")

outputs = model.generate(**inputs, max_new_tokens=100)

# 3. Decode the output back into text
print(tokenizer.decode(outputs[0], skip_special_tokens=True))

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


Solve using BODMAS: 8 + 3 * (6 - 2)/4Step 1: Brackets → (6 - 2) = 4
Step 2: Multiply → 3 * 4 = 12
Step 3: Division → 8 + 12 = 20
Final Answer: 20


THe above answer is incorrect !